# Chaining jobs

A recipe for chaining jobs on the Istari Digital Platform from Python: register a file as a Model, run an extraction job, feed one of its outputs into a second job, and walk the lineage of the final result.

It uses [`istari_fluent`](../fluent), an opinionated, chainable wrapper over the official [`istari-digital-client`](https://docs.istaridigital.com/developers/SDK/01-setup). The concepts (Systems, Models, Jobs, Products, Resources) are identical to the platform UI &mdash; `istari_fluent` just packages them behind entity-oriented methods.

### What we cover

- Connecting to the platform with a Personal Access Token.
- Registering a file as a Model.
- Running an extraction job and inspecting its products.
- Downloading an output and chaining it into a second job.
- Tracing the backward lineage of a revision.

### Prerequisites

- An **Istari Digital Platform account** and a **Personal Access Token**. See the [Sign-up Guide](https://docs.istaridigital.com/users/account/sign-up) and [Personal Access Tokens](https://docs.istaridigital.com/users/user-guide/settings#developer-settings--personal-access-tokens).
- An agent with the **Open Spreadsheet** integration and access to `@istari:extract`. See [Manage Tool Access](https://docs.istaridigital.com/users/admin-guide/user-management#manage-tool-access-for-a-user).
- The `Group3-UAS-Requirements.xlsx` sample file next to this notebook (already present in `samples/`).

### 1 &middot; Credentials

Create a `.env` file **next to this notebook** (so `samples/.env`) with the same two variable names the official Python client uses:

```
ISTARI_REGISTRY_URL=https://...paste your platform's registry URL here...
ISTARI_PERSONAL_ACCESS_TOKEN=...paste your token here...
```

You can find both in the platform under **Settings &rarr; Developer Settings**. Treat the token as a secret &mdash; it grants API access as you. Don't commit it, don't paste it into chat, don't screenshot it.

> **On a corporate network with an internal CA?** Pass the bundle into `from_env()` (or set `ISTARI_CA_BUNDLE` in your `.env`). See the commented example in the connect cell below.

### 2 &middot; Install dependencies

From the repository root:

```bash
cd fluent
uv sync --extra experiment
```

This creates `fluent/.venv/` containing `istari_fluent`, `jupyter`, and `ipykernel`.

### 3 &middot; Register the venv as a Jupyter kernel

A bare virtualenv is not auto-discovered by VS Code / Cursor. Register it once:

```bash
# still inside fluent/
uv run python -m ipykernel install --user --name istari-fluent --display-name "Python (istari_fluent)"
```

Then reload this notebook's kernel picker (top-right) and select **"Python (istari_fluent)"**.

To remove the kernel later: `jupyter kernelspec uninstall istari-fluent`.

> **A note on `istari_fluent`** &mdash; this is a productivity layer maintained alongside the official SDK. It is not the officially supported client. For production integrations, keep the core [`istari-digital-client`](https://docs.istaridigital.com/developers/SDK/01-setup) as your source of truth; use `istari_fluent` to prototype, explore, and build notebooks faster.

## 1 &middot; Connect and verify

`IstariPlatform.from_env()` reads `ISTARI_REGISTRY_URL` and `ISTARI_PERSONAL_ACCESS_TOKEN` (the same names used by the official Python client) and returns an `IstariPlatform` object.

The token is issued to *you*, so every call the notebook makes acts **on your behalf** &mdash; the same authorization rules you see in the UI apply here. If something is denied by the API, the cause is typically the same as in the UI; see [Sharing and access](https://docs.istaridigital.com/users/user-guide/sharing-and-access).

After connecting we run a quick **readiness check**: one round-trip that confirms the platform is reachable and your token is accepted.

In [ ]:
from pathlib import Path

from istari_fluent import IstariPlatform, JobDefinition

platform = IstariPlatform.from_env()

# Corporate network with an internal CA bundle? Point from_env() at your .pem:
# platform = IstariPlatform.from_env(ca_bundle="/path/to/ca.pem")
#
# Or set ISTARI_CA_BUNDLE in your .env / shell and just call from_env() above.

# Readiness check: one round-trip to confirm the platform answers and the token is accepted.
report = platform.client.readiness_check()
assert report.healthy, f"Platform reports unhealthy: {report}"

print(platform)

## 2 &middot; Register the spreadsheet as Model Resource

In Istari terminology, registering a local file creates a **Resource** of type **model**. Each resource has a stable identity with a **revision** history. 

The `Group3-UAS-Requirements.xlsx` sample is already next to this notebook. We also tag the resource with an `external_id` for reference.

In [ ]:
XLSX_PATH = Path.cwd() / "Group3-UAS-Requirements.xlsx"
EXTERNAL_ID = "fluent-tutorial-uas-requirements"
DISPLAY_NAME = "Group3-UAS-Requirements (tutorial)"

model = platform.upload_model(
    XLSX_PATH,
    external_id=EXTERNAL_ID,
    display_name=DISPLAY_NAME,
)
print(f"Uploaded new model {model.id}")

print(model)

## 3 &middot; Run the first extraction job

A **Job** is an instruction to the platform to run a specific **function** from a **tool** against a Model revision. Here we run `@istari:extract` with the `open_spreadsheet` tool &mdash; the same function you would select under **Jobs &rarr; Create Job** in the UI, and the same one used by the [Python Client 201 tutorial](https://docs.istaridigital.com/tutorials/python-client/201).

On this spreadsheet the extraction writes four artifacts:

| Artifact | What it contains |
|---|---|
| `named_cells.json` | Values of named ranges (`SubTitle`, `max_weight_value`, ...) |
| `worksheet_data.json` | Full cell data per sheet |
| `workbook.pdf` | Rendered PDF view of the workbook |
| `workbook.html` | Rendered HTML view |

We'll use the two-step pattern below so you can watch the status evolve:

1. `model.submit_job(definition)` returns a `JobView` immediately &mdash; the job is queued on the platform but not yet finished.
2. `job.wait(on_poll=...)` blocks until the job reaches a terminal state, calling the `on_poll` callback on every poll (default every 5 s) with the refreshed `JobView`. Any callable works &mdash; we pass a lambda that prints one line per poll. Swap in `logger.info`, a `tqdm` updater, a websocket push, etc., for richer feedback.
3. `.on_success()` raises `RuntimeError` if the job ended in `FAILED`, so a plain "no exception" means success.

> **Shortcut:** if you don't need live feedback, `model.run_job(definition)` submits + waits + checks success in one call. It also accepts `on_poll=...` if you want to mix and match.

In [ ]:
extract = JobDefinition(
    function="@istari:extract",
    tool_name="open_spreadsheet",
)

job1 = model.submit_job(extract)
print(f"Submitted job {job1.id}; polling...")

job1.wait(
    timeout=600,
    on_poll=lambda j: print(f"  [{j.status}] id={j.id}"),
).on_success()

print(f"\nJob 1 finished: {job1.status}")

# One-liner alternative: model.run_job(extract, timeout=600, on_poll=print)
# `run_job` == `submit_job` + `wait().on_success()`, with the same on_poll hook.

## 4 &middot; Inspect the products

Every job records the exact artifact revisions it wrote. `job1.get_products()` reads `job.revision.products` and returns a list of `ResourceView` objects **pinned to those revisions**. This matters: if Job 2 later writes a new revision of `named_cells.json`, the pinned view from Job 1 still points to the original bytes &mdash; just like the Resources tab for a specific job in the UI.

Look at the print-out below: every product has a `file_id` **and** a `rev_id`. The file id is the stable identity of the artifact file on the platform; the revision id is the specific version this job produced.

In [ ]:
products_1 = job1.get_products()
print(f"Job 1 wrote {len(products_1)} products:\n")
for p in products_1:
    print(f"  - {p.type:10s}  name={p.name!r:30s}  file={p.file_id}  rev={p.revision_id}")

## 5 &middot; Pick a product and download it

Once we have a pinned `ResourceView`, we can read the content in whatever shape is most convenient:

- `read_bytes()` &mdash; raw bytes
- `read_text()` &mdash; decoded text
- `read_json()` &mdash; parsed JSON (shortcut for `json.loads(view.read_text())`)
- `download(dir_or_path)` &mdash; save to disk

Because the view is pinned, every one of those calls returns the **same** revision even if newer revisions land later.

Here we load `named_cells.json` straight into a Python dict with `read_json()` and also save a copy next to the notebook &mdash; the same flow used in [Python Client 201 &sect; Register and run the first extraction](https://docs.istaridigital.com/tutorials/python-client/201#register-the-model-and-run-the-first-extraction).

In [ ]:
named_cells = job1.find_product(filename="named_cells.json")
assert named_cells is not None, "named_cells.json not produced"

data = named_cells.read_json()
print(f"named_cells.json has {len(data)} named ranges")
print("First 3 keys:", list(data)[:3])

download_path = named_cells.download(Path.cwd() / "outputs.json")
print(f"Downloaded to: {download_path}")

## 6 &middot; Chain a second job

Real workflows rarely stop at one job. The natural chaining pattern: **pick a Resource that Job 1 produced, then run another job on it.**

Job 1 wrote a `workbook.xlsx` Resource (an Artifact &mdash; a normalised copy of the input spreadsheet). We grab that Resource and call `run_job(...)` on it. Same method you used on `model` in Step 3 &mdash; any Resource accepts `run_job`. Step 7 will show the full chain from the final output back to the original upload.

> **Note on the tool:** in a real workflow this second job would usually invoke a *different* tool (e.g. a solver that consumes the extracted spreadsheet). For a self-contained tutorial we re-run `@istari:extract` with `open_spreadsheet` so you don't need to install anything else &mdash; the chaining mechanics are identical.

In [ ]:
workbook_xlsx = job1.find_product(filename="workbook.xlsx")
assert workbook_xlsx is not None, "Job 1 should have produced workbook.xlsx"
print(f"Chaining off: {workbook_xlsx}")

# Reuse the same JobDefinition as Job 1 for this self-contained example.
# In a real workflow you would typically pick a different tool/function here.
extract2 = JobDefinition(
    function="@istari:extract",
    tool_name="open_spreadsheet",
)

job2 = workbook_xlsx.run_job(
    extract2,
    timeout=600,
    on_poll=lambda j: print(f"  [{j.status}] id={j.id}"),
)
print(f"\nJob 2 finished: {job2.status}")

products_2 = job2.get_products()
print(f"\nJob 2 wrote {len(products_2)} products:")
for p in products_2:
    print(f"  - {p.type:10s}  name={p.name!r:30s}  rev={p.revision_id}")

## 7 &middot; Trace the lineage

The payoff of uploading, running jobs, and promoting artifacts is that **every revision knows how it got there**. `get_lineage()` walks backward from any revision and classifies each step:

| Step | Meaning |
|---|---|
| `upload` | A fresh file &mdash; no sources, the root of a chain |
| `job_run` | Produced by a Job |
| `promotion` | A Model promoted from another revision (relationship `promoted_from`) |
| `derived` | Any other derivation |

Below we pick `named_cells.json` from Job 2 and print the full chain back to the original upload. Expect something like:

```
- Artifact 'named_cells.json'
    step=job_run
  - Job '@istari:extract (<job2-id>)'
      step=job_run
    - Model 'workbook'
        step=promotion
      - Artifact 'workbook.xlsx'  [via promoted_from]
          step=job_run
        - Job '@istari:extract (<job1-id>)'
            step=job_run
          - Model 'Group3-UAS-Requirements (tutorial)'
              step=upload
```

Read top-down: Job 2 produced `named_cells.json`; Job 2 ran on a `workbook` Model linked back to the `workbook.xlsx` Resource; `workbook.xlsx` was produced by Job 1, which ran on the original upload.

In [ ]:
final_output = job2.find_product(filename="named_cells.json")
assert final_output is not None

tree = final_output.get_lineage(max_depth=6)
print("Lineage for Job 2's named_cells.json:\n")
tree.print_tree()

## Verify in the UI

 Sign in to the same platform you used for your token and cross-check:

1. **Files / Models** &mdash; You should see the file with DISPLAY_NAME we have used. Select it, you can verify the Model id and revision ids to the printout from Step 2.
2. **Jobs / Activity** &mdash; Two **Completed** extractions for `open_spreadsheet` / `@istari:extract`. The second one is linked back to the `workbook.xlsx` Resource from Step 6.
3. **Resources** &mdash; On each job, confirm the produced artifacts (`named_cells.json`, `worksheet_data.json`, `workbook.pdf`, `workbook.html`). Revision ids match the `rev=` values printed in Steps 4 and 6.




## Optional &middot; Archive the model

Archiving hides the Model from default listings in the UI and API. The data and lineage stay intact &mdash; this is a soft delete you can reverse later. Run this when you're done experimenting and don't want the tutorial Model cluttering your workspace.

In [ ]:
model.archive()

In [ ]:
import sys, istari_fluent
print(sys.executable)
print(istari_fluent.__file__)